# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## PCA: de tres variables a dos componentes
Datos sintéticos X1, X2 y X3. Tras escalar, PCA combina variables. Lee los porcentajes calculados en esta ejecución: no supongas 70 % o 90 % fijos. Varianza capturada no equivale a accuracy ni garantiza conservar lo relevante para una tarea.

In [ ]:
# ================================================================
# REDUCCIÓN DE DIMENSIONALIDAD CON PCA (Colab listo para pegar)
# Caso simple: dataset sintético de 3 variables
# ================================================================
# ¿Qué hace este notebook?
# 1) Genera un dataset con 3 variables correlacionadas.
# 2) Visualiza los datos en 3D.
# 3) Aplica PCA para reducir a 2D y a 1D.
# 4) Explica cómo leer los resultados: varianza explicada, componentes, proyecciones.
#
# ------------------------------------------------
# ¿Qué es PCA?
# ------------------------------------------------
# PCA = Principal Component Analysis.
# Objetivo: encontrar nuevas variables (componentes principales) que:
# - Son combinaciones lineales de las originales.
# - Están ordenadas por cuánta varianza (información) explican.
# - Permiten representar los datos en menos dimensiones perdiendo lo mínimo.
#
# Ejemplo:
# - Si lluvia y humedad están muy correlacionadas, PCA crea una sola
#   componente que resume ambas, reduciendo redundancia.
# ================================================================

from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D

# ----------------------------
# 1) Dataset sintético
# ----------------------------
# Creamos 3 variables con correlaciones:
# - X1 ~ normal
# - X2 ~ X1 con ruido (correlación alta)
# - X3 ~ independiente
rng = np.random.default_rng(42)
n = 200
X1 = rng.normal(50, 10, n)
X2 = X1 * 0.8 + rng.normal(0, 5, n) + 10
X3 = rng.normal(30, 8, n)

df = pd.DataFrame({"X1": X1, "X2": X2, "X3": X3})
display(df.head())

# ----------------------------
# 2) Visualización en 3D
# ----------------------------
fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df["X1"], df["X2"], df["X3"], alpha=0.6)
ax.set_xlabel("X1")
ax.set_ylabel("X2")
ax.set_zlabel("X3")
ax.set_title("Datos originales en 3D")
plt.show()

# ----------------------------
# 3) Escalado
# ----------------------------
# PCA es sensible a escala → StandardScaler.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# ----------------------------
# 4) PCA
# ----------------------------
pca = PCA(n_components=3)  # tantas como originales para analizar
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por cada componente:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"Componente {i+1}: {v:.2%}")

print("\nVarianza acumulada:", np.cumsum(pca.explained_variance_ratio_))

print("\nComponentes (pesos de cada variable original en cada PC):")
print(pd.DataFrame(pca.components_, columns=df.columns, index=[f"PC{i+1}" for i in range(3)]))

# ----------------------------
# 5) Visualización PCA 2D
# ----------------------------
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Datos reducidos a 2D con PCA")
plt.grid(True)
plt.show()

# ----------------------------
# 6) Visualización PCA 1D
# ----------------------------
plt.figure(figsize=(6,2))
plt.scatter(X_pca[:,0], np.zeros_like(X_pca[:,0]), alpha=0.6)
plt.xlabel("PC1 (única dimensión)")
plt.yticks([])
plt.title("Datos reducidos a 1D con PCA")
plt.show()

# ----------------------------
# 7) Interpretación
# ----------------------------
print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Varianza explicada:
   - PC1 captura la mayor proporción de varianza; consulta el porcentaje calculado.
   - PC2 añade otra parte significativa.
   - PC3 suele explicar poco si hay redundancia entre variables.
   → Decisión: consulta la varianza acumulada calculada para decidir cuántas componentes conservar.

2) Componentes (matriz de pesos):
   - Cada PC es combinación lineal de X1, X2, X3.
   - Si PC1 tiene coeficientes altos y parecidos en X1 y X2,
     significa que resume la "información común" entre ambas.

3) Gráfica 2D:
   - Los puntos en PC1 vs PC2 muestran casi la misma dispersión
     que en 3D, pero en un plano.

4) Gráfica 1D:
   - Comprimir a PC1 permite ordenar los datos en una sola dimensión
     sin perder demasiado de la estructura.

5) Uso práctico:
   - PCA reduce variables redundantes.
   - Útil para visualizar datos de alta dimensión o para mejorar
     el desempeño de otros algoritmos (clustering, regresión).
=============================================================
""")
